# SNOMED CT Term Lookup - Interactive Examples

This notebook demonstrates how to use the `SnomedTermLookup` class to find SNOMED CT codes by term matching.

## Setup

In [ ]:
import os
import sys

# Ensure project root is in Python path
CURRENT_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in locals()
    else os.getcwd()
)
PROJECT_ROOT = (
    os.path.abspath(os.path.join(CURRENT_DIR, ".."))
    if CURRENT_DIR.endswith("notebooks")
    else os.path.abspath(os.path.join(CURRENT_DIR, ".."))
)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.snomed_methods.snomed_term_lookup import create_term_lookup_from_directory

# Load UK SNOMED directory from environment variable or use default relative path
UK_SNOMED_DIR = os.environ.get(
    "UK_SNOMED_DIR",
    os.path.join(
        PROJECT_ROOT,
        "uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z",
    ),
)

# Initialize the term lookup with SNOMED CT UK Clinical RF2 data
lookup = create_term_lookup_from_directory(UK_SNOMED_DIR)
print("Loaded SNOMED data from: " + UK_SNOMED_DIR)
print("Number of concepts loaded: " + str(len(lookup.concept_ids)))

## 1. Basic Term Search

Find SNOMED CT concepts matching a specific term.

In [ ]:
# Search for terms containing 'meningioma' (default behavior)
results = lookup.find_concepts_by_term("meningioma", top_n=10)

print("Found " + str(len(results)) + " concepts matching 'meningioma':")
for i, (cui, term) in enumerate(results, 1):
    print(f"  {i}. {term} (CUI: {cui})")

## 2. Case-Insensitive Search

Search without worrying about case sensitivity.

In [ ]:
# Try different cases - all will match due to ignore_case=True (default)
cases = ["meningioma", "Meningioma", "MENINGIOMA"]

for case_term in cases:
    results = lookup.find_concepts_by_term(case_term, top_n=3)
    print(f"Search for '{case_term}': Found {len(results)} matches")

## 3. Batch Search

Search for multiple terms at once and see all results.

In [ ]:
# Define terms to search
terms_to_search = ["meningioma", "glioma", "tumor"]

# Perform batch search
batch_results = lookup.find_concepts_batch(terms_to_search, ignore_case=True)

print("Batch search results:")
# Group results by query term
current_term = None
for term, cui, matched in batch_results[:20]:
    if term != current_term:
        current_term = term
        print("  ")
        print("Query: '" + term + "'")
    print(f"    - {matched} (CUI: {cui})")

## 4. Fuzzy Search (Typo Tolerance)

Find concepts even when the search term has typos (requires `rapidfuzz`).

In [ ]:
# Check if rapidfuzz is available
try:
    import importlib.util

    HAS_FUZZY = importlib.util.find_spec("rapidfuzz") is not None
except ImportError:
    HAS_FUZZY = False

print("rapidfuzz available: " + str(HAS_FUZZY))

## 5. Get Concept Information

Retrieve detailed information about a specific concept.

In [ ]:
# First, find a meningioma concept
search_results = lookup.find_concepts_by_term("meningioma", top_n=1)

if search_results:
    cui = search_results[0][0]

    # Get detailed information
    info = lookup.getconcept_info(cui)

    print("Concept Information for CUI: " + str(cui))
    print("  Concept ID: " + info["concept_id"])
    if "preferred_name" in info:
        print("  Preferred Name: " + str(info["preferred_name"]))
    if "type_id" in info:
        print("  Type ID: " + str(info["type_id"]))
    if "synonym" in info:
        print("  Synonym: " + str(info["synonym"]))

## 6. Advanced: Combine with Existing SnomedRelations
Important: The `SnomedRelations` class requires a **MedCAT model** to retrieve concept names.

This notebook does not include MedCAT, so methods like:
- `expand_codes_parents_local()` - gets parent concepts
- `expand_codes_children_local()` - gets child concepts

will run but will return None for concept names. The concept codes (CUIs) are still available.

**What you need:**
1. Install and load a MedCAT model  
2. Initialize with `SnomedRelations(medcat=True, medcat_path='/path/to/medcat/model')`

See the code below for how to handle both scenarios.

In [ ]:
# Import SnomedRelations if available
try:
    import importlib.util

    HAS_SNOMED = importlib.util.find_spec("snomed_methods_v1") is not None
except ImportError:
    HAS_SNOMED = False

print("SnomedMethods v1 available: " + str(HAS_SNOMED))

## 7. Custom Search Parameters

Explore different search options.

In [ ]:
# Different search configurations
search_configs = [
    {"term": "meningioma", "match_prefix": False, "top_n": 5},
    {"term": "meningioma", "match_prefix": True, "top_n": 5},
    {"term": "meningioma", "ignore_case": True, "top_n": 10},
]

for i, config in enumerate(search_configs):
    results = lookup.find_concepts_by_term(**config)
    print(
        "Config {} - match_prefix={}, top_n={}: Found {} matches".format(
            i + 1, config.get("match_prefix", "N/A"), config["top_n"], len(results)
        )
    )

## 8. Real-World Example: Find All Tumor Types

Find all SNOMED concepts related to tumors and categorize them.

In [ ]:
# Search for various tumor-related terms
tumor_terms = ["carcinoma", "adenoma", "sarcoma", "melanoma", "lymphoma"]

print("Tumor-related term search results:")
for term in tumor_terms:
    results = lookup.find_concepts_by_term(term, top_n=5)
    print("")
    print(f"'{term}': Found {len(results)} matches")
    for cui, display_term in results[:3]:
        print(f"  - {display_term} (CUI: {cui})")

## Next Steps

For more advanced usage, see:
- `examples/example_term_lookup.py` - More detailed examples
- `TERM_LOOKUP_README.md` - Complete API documentation